# Hello World — Claude Managed Agents

This notebook walks through the minimal end-to-end flow for Claude Managed Agents:

```
Environment (once) → Agent (once) → Session (every run) → Stream events
```

| Object | Lifecycle | Purpose |
|--------|-----------|----------|
| **Environment** | Create once | Sandboxed container where tools run (bash, files, code) |
| **Agent** | Create once | Versioned config: model, system prompt, tools |
| **Session** | Create per run | Links agent + environment; Anthropic runs the loop |

> **In production:** persist `environment.id` and `agent.id` — don't recreate them on every run.

## 1. Setup

Install dependencies and load the API key from the `.env` file in the project root.

`find_dotenv()` walks up from the current directory until it finds a `.env` file, so it works regardless of where the notebook is located.

In [6]:
%pip install -q -U anthropic python-dotenv --user


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # finds .env in the project root

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
print("Client ready")

Client ready


## 2. Create an Environment

An environment is the **sandboxed container** where tools execute. Think of it as the machine Claude will use to run bash commands, read/write files, and execute code.

- `type: "cloud"` — Anthropic manages the infra (alternative: `self_hosted`)
- `networking.type: "unrestricted"` — allows outbound internet access

**Create this once and reuse the ID.** Creating a new environment per session is wasteful.

In [8]:
environment = client.beta.environments.create(
    name="basics-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID : {environment.id}")
print(f"Created at     : {environment.created_at}")

Environment ID : env_01NbhfRNEE8aSByC8mydKsCX
Created at     : 2026-05-24T01:46:04.676201Z


## 3. Create an Agent

An agent is a **persisted, versioned config** object. It stores:
- `model` — which Claude model to use
- `system` — the system prompt
- `tools` — what tools Claude can invoke

`agent_toolset_20260401` is Anthropic's prebuilt toolset: bash, file operations, code execution, and more — all running inside the environment container.

Every time you update an agent, a new **immutable version** is created. Sessions pin to a specific version, so updates never break running sessions.

In [9]:
agent = client.beta.agents.create(
    name="Hello World Agent",
    model="claude-opus-4-7",
    system="You are a helpful assistant. Keep your answers concise.",
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": True},
        }
    ],
)

print(f"Agent ID  : {agent.id}")
print(f"Version   : {agent.version}")
print(f"Model     : {agent.model}")

Agent ID  : agent_01DYwVkJc1mHHxS6Fqps26zt
Version   : 1
Model     : BetaManagedAgentsModelConfig(id='claude-opus-4-7', speed='standard')


## 4. Create a Session

A session is a **single run** — it links an agent to an environment. Anthropic's orchestration layer runs the agent loop on its servers; you just send events and receive results over SSE.

Note: we pin to the exact `agent.version` so that future agent updates don't affect this session.

In [10]:
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
)

print(f"Session ID : {session.id}")
print(f"Status     : {session.status}")

Session ID : sesn_0175hLMuvpckB7n46d3LibvG
Status     : idle


## 5. Stream Events

This is the **stream-first pattern** — critical to follow in order:

1. **Open the stream first** (`events.stream(session_id)`) — starts listening for SSE events
2. **Then send your message** (`events.send(...)`) — triggers the agent loop
3. **Iterate over the stream** — handle events as they arrive

If you send before opening the stream, you'll miss early events.

### Key event types

| Event type | Meaning |
|---|---|
| `agent.message` | Claude's response (may contain text, tool calls, etc.) |
| `session.status_idle` | Agent finished its turn; you can send another message |
| `session.status_terminated` | Session is done; no more messages possible |

In [11]:
# Open the stream BEFORE sending any events
with client.beta.sessions.events.stream(session.id) as stream:

    # Send the user message to trigger the agent loop
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": "Say hello and tell me one fun fact about octopuses.",
                    }
                ],
            }
        ],
    )

    # Process events as they stream in
    print("Agent: ", end="", flush=True)
    for event in stream:
        if event.type == "agent.message":
            for block in event.content:
                if block.type == "text":
                    print(block.text, end="", flush=True)

        elif event.type == "session.status_idle":
            break  # agent finished its turn

        elif event.type == "session.status_terminated":
            break  # session ended

print()  # newline after streamed output

Agent: Hello! 🐙

Fun fact: Octopuses have three hearts — two pump blood through the gills, and the third pumps it to the rest of the body. Oddly, the main heart actually stops beating when they swim, which is why they often prefer crawling!


## Summary

You just ran a full Claude Managed Agents flow:

```
environments.create()  →  store environment.id
agents.create()        →  store agent.id + agent.version
sessions.create()      →  new session per run
events.stream()        →  open stream first
events.send()          →  send message, triggers loop
for event in stream    →  handle agent.message until status_idle
```

### Next steps
- **02-multi-turn** — keep the session alive and send follow-up messages
- **03-tools** — use bash and file tools inside the container
- **04-mcp** — connect external MCP servers to the agent